## Imports

In [1]:
# !pip install osmnx --user
# !pip install folium --user
# !pip install folium matplotlib mapclassify

import re
import os 
import osmnx
import folium
import geopandas as gpd 

import pandas as pd
import numpy as np
import seaborn as sns
import pyarrow
sns.set()
import gc
gc.collect()
import matplotlib.pyplot as plt  
# plt.style.use('seaborn-notebook')
# COLOR = 'white'
# plt.rcParams['text.color'] = COLOR
# plt.rcParams['axes.labelcolor'] = COLOR
# plt.rcParams['xtick.color'] = COLOR
# plt.rcParams['ytick.color'] = COLOR
%matplotlib inline
from tqdm import tqdm
tqdm.pandas()
import datetime

from pyhive import presto
from datetime import datetime, timedelta, date
import warnings
warnings.filterwarnings("ignore")
import math
from functools import reduce

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

### connection

In [2]:
import requests
def get_processing_url():
    presto_gateway = "http://presto-gateway.processing.data.production.internal/gateway/backend/active"
    active_backends = requests.get(presto_gateway).json()
    processing_backend = next(filter(lambda backend: backend['routingGroup'] == 'processing', active_backends),
                              None)
    return processing_backend['proxyTo'].replace("http://", "").rstrip("/")


connection = presto.connect(
    # host="bi-presto.serving.data.production.internal",
    # host="insights-download.serving.data.production.internal",
    # host = "bi-trino-2.serving.data.production.internal",
    host = "bi-trino.serving.data.production.internal",
    # host="presto.processing.yoda.run",
    # host='presto-gateway.processing.data.production.internal',
    # host = "processing-2.processing.data.production.internal",
    # host = "processing.processing.data.production.internal",
#     host = get_processing_url(),
    port=80,
    protocol="http",
    catalog="hive",
    # username="sameer.narkhede@rapido.bike",
#     username = "airflow-user",
    username = "manoj.ravirajan@rapido.bike",
    
)


def run_query(query):
    return pd.read_sql_query(query, connection)

## OSM

City -> AOI to be passes in OSM <br>
Bangalore -> ['Bangalore, India'] <br>
Chennai -> ['Chennai District, India', 'Tambaram District, India'] <br>
Hyderabad -> ['Hyderabad, India'] <br>
Kolkata -> ['Kolkata District, India'] <br>
Delhi -> ['Delhi, India'] <br>
Jaipur -> ['Jaipur District, India'] <br>
Pune -> ['Pune, India'] <br>
Indore -> ['Indore District, India'] <br>
Ahmedabad -> ['Ahmedabad District, India'] <br>
Nagpur -> ['Nagpur, India'] <br>
Lucknow -> ['Lucknow District, India'] <br>
Coimbatore -> ['Coimbatore District, India'] <br>
Guwahati -> ['Guwahati District, India'] <br>
Vishakapatnam -> ['Visakhapatnam, India'] <br>
Mumbai -> ['Bombay, India', 'Mumbai, India', 'Mumbai Suburban, India', 'Thane, India', 'Navi Mumbai, India'] <br>
Bhopal -> ['Bhopal District, India'] <br>
Tirupati -> ['Tirupati District, India'] <br>
Patna -> ['Patna District, India'] <br>

In [3]:
# Bangalore District, India
AOI = ['Delhi, India']

aoi_gdf = osmnx.geocode_to_gdf(AOI)
aoi_gdf
#plot area of interest
basemap = aoi_gdf.explore(color='lightblue')
basemap

### User-defined functions

In [5]:
!pip install h3
import shapely, h3
from shapely.geometry.polygon import Polygon
from shapely.geometry.multipolygon import MultiPolygon
from shapely.geometry.point import Point
from shapely.geometry.linestring import LineString
from shapely.geometry import mapping

def geometry_to_hex(geometry,hex_resolution):
    hex_12_ids=set()
    
    try:
    
        if type(geometry) == MultiPolygon:
            geojson = [shapely.geometry.mapping(item) for item in geometry]
            hex_id = [h3.polyfill(item,hex_resolution) for item in geojson]
            for item in hex_id:
                hex_12_ids=hex_12_ids.union(item)

        elif type(geometry) == shapely.geometry.polygon.Polygon:
            geojson = shapely.geometry.mapping(geometry)
            hex_id = h3.polyfill(geojson,hex_resolution)
            hex_12_ids=hex_12_ids.union(hex_id)

        elif type(geometry) == Point:
            hex_id=h3.geo_to_h3(geometry.x, geometry.y, hex_resolution)
            hex_12_ids={hex_id}
        
        ## This for loop forms a boundary around an area (say national park) and find out boundry around that area by taking all the hex_12 ids 
        # for item in hex_12_ids:
        #     hex_12_ids=hex_12_ids.union(h3.k_ring(item,1))
        return(hex_12_ids)
    
    except:
        pass

### geometries_from_polygon

In [6]:
educational = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'buildings': ['school','university'],
                                              'amenity':['school','university','college'],
                                              'landuse':['education','school']})

#educational.explore(color = 'cyan',m= basemap)
educational=educational.reset_index()
educational['usecase']='educational'
educational.shape

(1104, 108)

In [7]:
place_of_worship = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'building': ['temple','mosque','shrine','cathedral','chapel','synagogue','church'],
                                              'amenity':'place of worship'})

#place_of_worship.explore(color = 'green',m= basemap)

place_of_worship=place_of_worship.reset_index()
place_of_worship['usecase']='place_of_worship'
place_of_worship.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/241563652.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  place_of_worship = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(68, 65)

In [8]:
leisure = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'leisure':True,
                                              'landuse':['beer_graden','recreation_ground','recreational','recreational_area'],
                                              'amenity':['arts_centre','casino','cinema','dive_centre','community_centre','brothel','internet_cafe','courthouse','cinema','sauna'],
                                              'building':'stadium'})

#leisure.explore(color = 'orange',m= basemap)
leisure=leisure.reset_index()
leisure['usecase']='leisure'
leisure.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/3191018952.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  leisure = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(4552, 162)

In [9]:
transit_station = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'railway':['station','junction','terminal', 'terminus'],
                                              'amenity':'bus_station',
                                              'highway':['bus_stand','platform'],
                                              'public_transport':'station',
                                             'landuse':'depot'
                                             })

#transit_station.explore(color = 'red',m= basemap)
transit_station=transit_station.reset_index()
transit_station['usecase']='transit_station'
transit_station.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/718829431.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  transit_station = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(338, 97)

In [21]:
display(transit_station.railway.unique())
display(transit_station.network.unique())
display(transit_station.operator.unique())
display(transit_station.public_transport.unique())
display(transit_station.bus.unique())
display(transit_station.amenity.unique())

array(['station', nan, 'platform'], dtype=object)

array(['Delhi Metro', 'IR', nan, 'Delhi Public Transport', 'DTC',
       'Pink Line'], dtype=object)

array(['Delhi Metro Rail Corporation Limited', 'NR', nan, 'IIT Delhi',
       'Delhi Transport Corporation', 'DTC',
       'Delhi Transport Corporation (DTC), Delhi Transit', 'DMRC'],
      dtype=object)

array(['station', nan], dtype=object)

array([nan, 'yes'], dtype=object)

array([nan, 'bus_station'], dtype=object)

#### Check the values (transit_station)

In [22]:
r = transit_station[transit_station['railway'] == 'station']
r['network'].unique()

array(['Delhi Metro', 'IR', nan], dtype=object)

In [12]:
ir = transit_station[transit_station['network'] == 'IR']
ir['name'].unique()

array(['Narela', 'Holambi Kalan', 'Khera Kalan', 'Delhi Cantonment',
       'Delhi Shahdara Junction', 'Vivek Vihar', 'Delhi Safdarjang',
       'Sewa Nagar', 'Okhla', 'Subzimandi', 'Delhi Junction',
       'Delhi Azadpur', 'Dayabasti', 'Patel Nagar', 'Delhi Sarai Rohilla',
       'Adarsh Nagar', 'Palam', 'Bijwasan', 'Anand Vihar Terminus',
       'Shakurbasti', 'Tughlakabad', 'Brar Square', 'Delhi Kishanganj',
       'Lajpat Nagar', 'Lodhi Colony', 'Pragati Maidan', 'Sarojini Nagar',
       'Shivaji Bridge', 'Tilak Bridge', 'Vivekanand Puri',
       'Hazrat Nizamuddin Junction', 'Shahabad Mohammadpur', 'Ghevra',
       'Nangloi', 'Sadar Bazar', 'New Delhi', 'Nasirpur'], dtype=object)

In [23]:
m = transit_station[transit_station['network'] == 'Delhi Metro']
m['operator'].unique()

array(['Delhi Metro Rail Corporation Limited'], dtype=object)

In [24]:
l = transit_station[transit_station['amenity'] == 'bus_station']
print((l.name.isna().sum() * 100)/l.shape[0])
l['name'].unique()

16.923076923076923


array([nan, 'Mori Gate Bus Terminal', 'Hari Nagar Bus Depot',
       'Princess Park', 'Safdarjung Terminal', 'Pusa Institute',
       'DTC Bus Depot, Dichaon Kalan', 'Aya Nagar Bus Terminal', 'Okhla',
       'Bikaner Bus Depot', 'Paharganj bus stand', 'Majnu Ka Tila',
       'Shivalik Zanskar BUS STOP', 'iit bus stop', 'Hospital BUS STOP',
       'kardampuri', 'Potala Ground', 'Bus Station', 'Badarpur Border',
       'Sanskrit Vidya Peeth BUS STOP', 'Western Court Bus Terminus',
       'Transfer bus station', 'Goldline Super Deluxe Bus Station',
       'Chirag Dilli Metro Station Gate no.2',
       'Akshardam Pillar 36 Bus Stand', 'Kashmere Gate ISBT',
       'Anand Vihar Interstate Bus Terminal', 'Uttam Nagar Terminal',
       'DTC Bus Depot Sector 2', 'Azadpur', 'DTC Bus Depot',
       'Sarai Kale Khan ISBT', 'Seemapuri Bus Depot', 'Dwarka Bus Depot',
       'Najafgarh Main Bus Stand', 'Kaushambi Bus Depot',
       'Hari Nagar DTC Depot', 'Delhi Transport Corporation',
       'Kair D

In [15]:
office = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'building':['office','commercial']})

#office.explore(color = 'black',m= basemap)
office=office.reset_index()
office['usecase']='office'
office.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/2755102806.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  office = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(431, 89)

In [16]:
health_and_personal = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'amenity':['dentist','hospital','gym','clinic','nursing home'],
                                              'building':['dentist','hospital'],
                                              'landuse':'healthcare'})

#health_and_personal.explore(color = 'yellow',m= basemap)
health_and_personal=health_and_personal.reset_index()
health_and_personal['usecase']='health_and_personal'
health_and_personal.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/3219860144.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  health_and_personal = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(1160, 94)

In [17]:
food = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'amenity':['bar','pub','resturant','fast_food','food_court','cafe','juice bar','kitchen','ice cream']})

#food.explore(color = 'brown',m= basemap)
food=food.reset_index()
food['usecase']='food'
food.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/687420186.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  food = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(867, 126)

In [18]:
residential = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'landuse' : 'residential'})
# residential1 = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
#                                       tags = {'building':['appartments','residential'],'highway':['residential']})

 
#residential.explore(color = 'grey',m= basemap)
residential=residential.reset_index()
residential['usecase']='residential'
residential.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/220764997.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  residential = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(2760, 66)

In [19]:
household_needs = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'amenity':['car_rental','car_wash','fuel','bank'],
                                              'building':['civic','retail','kiosk']
                                              })

#household_needs.explore(color = 'olive',m= basemap)
household_needs=household_needs.reset_index()
household_needs['usecase']='household_needs'
household_needs.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/407405380.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  household_needs = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(1306, 122)

In [20]:
govt_amenity = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,
                                      tags = {'amenity':['fire_station','grave_yard','crematorium','fire hydrant','funeral hall','police','post office','post box','post depot'],
                                              'building':['police_station','post_office','passport_office']})

#govt_amenity.explore(color = 'magenta',m= basemap)
govt_amenity=govt_amenity.reset_index()
govt_amenity['usecase']='govt_amenity'
govt_amenity.shape

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/723630929.py:1: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  govt_amenity = osmnx.geometries_from_polygon(polygon = aoi_gdf.geometry.geometry.unary_union,


(195, 36)

### Concat

In [26]:
# Concatenating all the usecases in a single df
city_usecase = pd.concat([educational, place_of_worship, leisure, transit_station,office,health_and_personal, food, residential,household_needs,govt_amenity], axis=0)
city_usecase.head()

,element_type,osmid,amenity,created_by,name,geometry,addr:postcode,addr:street,country,name:en,name:hi,phone,website,barrier,ref,addr:housenumber,source,name:etymology:wikidata,wikidata,wikipedia,addr:housename,alt_name,addr:block,addr:place,addr:city,operator,building,addr:district,email,grades,religion,start_date,wikimedia_commons,office,opening_hours,description,shop,fax,phone_1,branch,wheelchair,isced:level,operator:type,addr:subdistrict,contact:phone,name:ru,contact:email,female,contact:fax,contact:website,short_name,brand,brand:wikidata,school:gender,survey:date,addr:suburb,note,addr:hamlet,contact:mobile,addr:full,check_date,nodes,name:ar,name:ml,name:zh,addr:country,landuse,postal_code,old_name,name:ur,male,internet_access,internet_access:fee,internet_access:ssid,name:ta,government,designation,is_in,boundary,addr:county,alt_name_1,int_name,tourism,not:brand:wikidata,name:fr,name:mr,addr:city:ur,leisure,addr:locality,building:levels,addr:unit,addr:neighbourhood,fixme,wall,level,building:colour,roof:shape,ways,admin_level,type,addr:sector,contact:facebook,dogs,internet_access:operator,not:name,surveillance,surveillance:type,usecase,denomination,name:de,name:es,name:he,name:it,name:pt,name:uk,building:material,height,is_in:city,is_in:country,is_in:country_code,is_in:iso_3166_2,length,name:bn,name:gu,name:ja,name:pa,name:te,width,name:fa,layer,access,fee,natural,sport,highway,bicycle,foot,facebook,key,lit,location,entrance,historic,motor_vehicle,community_centre:for,horse,brand:wikipedia,fitness_station,dance:teaching,unisex,landmark,air_conditioning,material,opening_hours:covid19,climbing:boulder,indoor,dog,location:transition,addr:khasra,image,contact:instagram,mobile,blind,garden:type,protect_class,surface,alt_name:hi,place,construction,disused:landuse,disused:name,disused:operator,covered,hoops,building:part,disused:amenity,water,grass,grassland,emergency,playground,fountain,addr:city:en,name:alt,lanes,oneway,segregated,smoking,source:boundary,area,residential,amenity_1,operator:short,operator:wikidata,playground:theme,roof:colour,swimming_pool,addr:province,addr:state,community_centre,notes,alt_name:ur,name:pnb,name:ro,colour,network,public_transport,railway,station,subway,name:kn,name:ks,train,network:wikidata,payment:cards,payment:cash,payment:coins,payment:contactless,payment:electronic_purses,alt_name:ar,alt_name:ks,name:az-Arab,name:azb,name:el,name:gr,name:grc,name:hy,name:hyw,name:ug,name_alt,long_name,long_name:ur,bus,depth,public_transport:version,toilets:wheelchair,bench,shelter,tactile_paving,wikipedia:ur,departures_board,operator:ur,line,elevation,official_name,depot,nohousenumber,popular name,payment:american_express,payment:amex,payment:credit_cards,payment:debit_cards,payment:diners_club,payment:discover_card,payment:maestro,payment:mastercard,payment:visa,payment:visa_debit,payment:visa_electron,diplomatic,embassy,target,roof:levels,building:levels:underground,cuisine,man_made,diet:lacto_vegetarian,diet:vegan,toilets,toilets:access,addr:street:ur,healthcare,healthcare:speciality,addr:floor,addr:place_1,phone_2,phone_3,health_facility:type,contact:nodal_officer_email,contact:nodal_officer_mobile,opening_hours:opd,emergency:phone,contact:youtube,addr:parentstreet,url,beds,loc_name,brand:pa,brand:ur,brand:wikipedia:pa,brand:wikipedia:ur,delivery,takeaway,brand:ks,indoor_seating,outdoor_seating,diet:vegetarian,wifi,alt_name:en,alt_name:pa,alt_name:pnb,brand:en,brand:hi,brand:pnb,official_name:pa,official_name:ur,drive_through,source:height,drink:coffee,drink:tea,seats,tables,payment:paytm,self_service,contact:snapchat,contact:twitter,happy_hours,alcohol,drink:hot_chocolate,organic,craft,cafe,fast_food,street_vendor,addr:place:ur,occupied,addr:plotnumber,informal,is_in:locality,atm,fuel:cng,payment:bhim,payment:mobikwik,payment:phonepe,payment:upi,compressed_air,fuel:diesel,fuel:octane_98,brand:website,source:amenity,fuel:petrol,fuel:octane_95,cash_withdrawal,brand:short,brand:kn,phone_4,branch:hi,branch:ty

### (lat, long) pair
OSM gives (long, lat) pair but our datasets have (lat, long) pair so reversing it

In [27]:
# OSM gives (long, lat) pair but our datasets have (lat, long) pair so reversing it
city_usecase['geometry_rev']=gpd.GeoSeries(city_usecase['geometry']).map(lambda polygon: shapely.ops.transform(lambda x, y: (y, x), polygon))

city_usecase['hex_12_id_set']=city_usecase['geometry_rev'].apply(lambda x: geometry_to_hex(x,hex_resolution=12))

city_usecase.dropna(subset = ['hex_12_id_set'], inplace=True)


city_usecase['hex_12_list']=city_usecase['hex_12_id_set'].apply(lambda x: list(x))
city_usecase=city_usecase.explode('hex_12_list')
city_usecase.head()

/Users/rapido/anaconda3/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/Users/rapido/anaconda3/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/Users/rapido/anaconda3/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

,element_type,osmid,amenity,created_by,name,geometry,addr:postcode,addr:street,country,name:en,name:hi,phone,website,barrier,ref,addr:housenumber,source,name:etymology:wikidata,wikidata,wikipedia,addr:housename,alt_name,addr:block,addr:place,addr:city,operator,building,addr:district,email,grades,religion,start_date,wikimedia_commons,office,opening_hours,description,shop,fax,phone_1,branch,wheelchair,isced:level,operator:type,addr:subdistrict,contact:phone,name:ru,contact:email,female,contact:fax,contact:website,short_name,brand,brand:wikidata,school:gender,survey:date,addr:suburb,note,addr:hamlet,contact:mobile,addr:full,check_date,nodes,name:ar,name:ml,name:zh,addr:country,landuse,postal_code,old_name,name:ur,male,internet_access,internet_access:fee,internet_access:ssid,name:ta,government,designation,is_in,boundary,addr:county,alt_name_1,int_name,tourism,not:brand:wikidata,name:fr,name:mr,addr:city:ur,leisure,addr:locality,building:levels,addr:unit,addr:neighbourhood,fixme,wall,level,building:colour,roof:shape,ways,admin_level,type,addr:sector,contact:facebook,dogs,internet_access:operator,not:name,surveillance,surveillance:type,usecase,denomination,name:de,name:es,name:he,name:it,name:pt,name:uk,building:material,height,is_in:city,is_in:country,is_in:country_code,is_in:iso_3166_2,length,name:bn,name:gu,name:ja,name:pa,name:te,width,name:fa,layer,access,fee,natural,sport,highway,bicycle,foot,facebook,key,lit,location,entrance,historic,motor_vehicle,community_centre:for,horse,brand:wikipedia,fitness_station,dance:teaching,unisex,landmark,air_conditioning,material,opening_hours:covid19,climbing:boulder,indoor,dog,location:transition,addr:khasra,image,contact:instagram,mobile,blind,garden:type,protect_class,surface,alt_name:hi,place,construction,disused:landuse,disused:name,disused:operator,covered,hoops,building:part,disused:amenity,water,grass,grassland,emergency,playground,fountain,addr:city:en,name:alt,lanes,oneway,segregated,smoking,source:boundary,area,residential,amenity_1,operator:short,operator:wikidata,playground:theme,roof:colour,swimming_pool,addr:province,addr:state,community_centre,notes,alt_name:ur,name:pnb,name:ro,colour,network,public_transport,railway,station,subway,name:kn,name:ks,train,network:wikidata,payment:cards,payment:cash,payment:coins,payment:contactless,payment:electronic_purses,alt_name:ar,alt_name:ks,name:az-Arab,name:azb,name:el,name:gr,name:grc,name:hy,name:hyw,name:ug,name_alt,long_name,long_name:ur,bus,depth,public_transport:version,toilets:wheelchair,bench,shelter,tactile_paving,wikipedia:ur,departures_board,operator:ur,line,elevation,official_name,depot,nohousenumber,popular name,payment:american_express,payment:amex,payment:credit_cards,payment:debit_cards,payment:diners_club,payment:discover_card,payment:maestro,payment:mastercard,payment:visa,payment:visa_debit,payment:visa_electron,diplomatic,embassy,target,roof:levels,building:levels:underground,cuisine,man_made,diet:lacto_vegetarian,diet:vegan,toilets,toilets:access,addr:street:ur,healthcare,healthcare:speciality,addr:floor,addr:place_1,phone_2,phone_3,health_facility:type,contact:nodal_officer_email,contact:nodal_officer_mobile,opening_hours:opd,emergency:phone,contact:youtube,addr:parentstreet,url,beds,loc_name,brand:pa,brand:ur,brand:wikipedia:pa,brand:wikipedia:ur,delivery,takeaway,brand:ks,indoor_seating,outdoor_seating,diet:vegetarian,wifi,alt_name:en,alt_name:pa,alt_name:pnb,brand:en,brand:hi,brand:pnb,official_name:pa,official_name:ur,drive_through,source:height,drink:coffee,drink:tea,seats,tables,payment:paytm,self_service,contact:snapchat,contact:twitter,happy_hours,alcohol,drink:hot_chocolate,organic,craft,cafe,fast_food,street_vendor,addr:place:ur,occupied,addr:plotnumber,informal,is_in:locality,atm,fuel:cng,payment:bhim,payment:mobikwik,payment:phonepe,payment:upi,compressed_air,fuel:diesel,fuel:octane_98,brand:website,source:amenity,fuel:petrol,fuel:octane_95,cash_withdrawal,brand:short,brand:kn,phone_4,branch:hi,branch:ty

In [40]:
city_usecase['network'].unique()

array([nan, 'Delhi Metro', 'IR', 'Delhi Public Transport', 'DTC',
       'Pink Line', 'DMRC'], dtype=object)

In [41]:
city_usecase['hex_12_list'].isna().sum()/city_usecase['hex_12_list'].shape[0]

0.00047249793138100775

In [42]:
city_usecase1 = city_usecase[~city_usecase['hex_12_list'].isna()]
city_usecase1['hex_8'] = city_usecase1['hex_12_list'].apply(lambda x: h3.h3_to_parent(x,8))
city_usecase1['hex_10'] = city_usecase1['hex_12_list'].apply(lambda x: h3.h3_to_parent(x,10))

/Users/rapido/anaconda3/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/rapido/anaconda3/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [43]:
city_usecase1['hex_12_list'].dtypes

dtype('O')

## Geo Use-case
### Metro

In [44]:
city_usecase1['network'].unique()

array([nan, 'Delhi Metro', 'IR', 'Delhi Public Transport', 'DTC',
       'Pink Line', 'DMRC'], dtype=object)

In [46]:
metro = city_usecase1[(city_usecase1['network']=='Delhi Metro')]
print(metro.name.isna().sum(), metro.shape, metro.hex_10.nunique())
metro.head()

0 (347, 369) 224


,element_type,osmid,amenity,created_by,name,geometry,addr:postcode,addr:street,country,name:en,name:hi,phone,website,barrier,ref,addr:housenumber,source,name:etymology:wikidata,wikidata,wikipedia,addr:housename,alt_name,addr:block,addr:place,addr:city,operator,building,addr:district,email,grades,religion,start_date,wikimedia_commons,office,opening_hours,description,shop,fax,phone_1,branch,wheelchair,isced:level,operator:type,addr:subdistrict,contact:phone,name:ru,contact:email,female,contact:fax,contact:website,short_name,brand,brand:wikidata,school:gender,survey:date,addr:suburb,note,addr:hamlet,contact:mobile,addr:full,check_date,nodes,name:ar,name:ml,name:zh,addr:country,landuse,postal_code,old_name,name:ur,male,internet_access,internet_access:fee,internet_access:ssid,name:ta,government,designation,is_in,boundary,addr:county,alt_name_1,int_name,tourism,not:brand:wikidata,name:fr,name:mr,addr:city:ur,leisure,addr:locality,building:levels,addr:unit,addr:neighbourhood,fixme,wall,level,building:colour,roof:shape,ways,admin_level,type,addr:sector,contact:facebook,dogs,internet_access:operator,not:name,surveillance,surveillance:type,usecase,denomination,name:de,name:es,name:he,name:it,name:pt,name:uk,building:material,height,is_in:city,is_in:country,is_in:country_code,is_in:iso_3166_2,length,name:bn,name:gu,name:ja,name:pa,name:te,width,name:fa,layer,access,fee,natural,sport,highway,bicycle,foot,facebook,key,lit,location,entrance,historic,motor_vehicle,community_centre:for,horse,brand:wikipedia,fitness_station,dance:teaching,unisex,landmark,air_conditioning,material,opening_hours:covid19,climbing:boulder,indoor,dog,location:transition,addr:khasra,image,contact:instagram,mobile,blind,garden:type,protect_class,surface,alt_name:hi,place,construction,disused:landuse,disused:name,disused:operator,covered,hoops,building:part,disused:amenity,water,grass,grassland,emergency,playground,fountain,addr:city:en,name:alt,lanes,oneway,segregated,smoking,source:boundary,area,residential,amenity_1,operator:short,operator:wikidata,playground:theme,roof:colour,swimming_pool,addr:province,addr:state,community_centre,notes,alt_name:ur,name:pnb,name:ro,colour,network,public_transport,railway,station,subway,name:kn,name:ks,train,network:wikidata,payment:cards,payment:cash,payment:coins,payment:contactless,payment:electronic_purses,alt_name:ar,alt_name:ks,name:az-Arab,name:azb,name:el,name:gr,name:grc,name:hy,name:hyw,name:ug,name_alt,long_name,long_name:ur,bus,depth,public_transport:version,toilets:wheelchair,bench,shelter,tactile_paving,wikipedia:ur,departures_board,operator:ur,line,elevation,official_name,depot,nohousenumber,popular name,payment:american_express,payment:amex,payment:credit_cards,payment:debit_cards,payment:diners_club,payment:discover_card,payment:maestro,payment:mastercard,payment:visa,payment:visa_debit,payment:visa_electron,diplomatic,embassy,target,roof:levels,building:levels:underground,cuisine,man_made,diet:lacto_vegetarian,diet:vegan,toilets,toilets:access,addr:street:ur,healthcare,healthcare:speciality,addr:floor,addr:place_1,phone_2,phone_3,health_facility:type,contact:nodal_officer_email,contact:nodal_officer_mobile,opening_hours:opd,emergency:phone,contact:youtube,addr:parentstreet,url,beds,loc_name,brand:pa,brand:ur,brand:wikipedia:pa,brand:wikipedia:ur,delivery,takeaway,brand:ks,indoor_seating,outdoor_seating,diet:vegetarian,wifi,alt_name:en,alt_name:pa,alt_name:pnb,brand:en,brand:hi,brand:pnb,official_name:pa,official_name:ur,drive_through,source:height,drink:coffee,drink:tea,seats,tables,payment:paytm,self_service,contact:snapchat,contact:twitter,happy_hours,alcohol,drink:hot_chocolate,organic,craft,cafe,fast_food,street_vendor,addr:place:ur,occupied,addr:plotnumber,informal,is_in:locality,atm,fuel:cng,payment:bhim,payment:mobikwik,payment:phonepe,payment:upi,compressed_air,fuel:diesel,fuel:octane_98,brand:website,source:amenity,fuel:petrol,fuel:octane_95,cash_withdrawal,brand:short,brand:kn,phone_4,branch:hi,branch:ty

### Bus Station

In [47]:
city_usecase1['amenity'].unique()

array(['school', 'college', 'university', nan, 'place_of_worship',
       'cinema', 'community_centre', 'arts_centre', 'courthouse',
       'internet_cafe', 'music_school', 'Bawana Bus depot', 'bus_station',
       'events_venue', 'restaurant', 'hospital', 'dentist', 'clinic',
       'veterinary', 'cafe', 'fast_food', 'bar', 'pub', 'food_court',
       'bank', 'fuel', 'car_rental', 'car_wash', 'fire_station',
       'grave_yard', 'police', 'crematorium'], dtype=object)

In [48]:
bus_station = city_usecase1[city_usecase1['amenity'] == 'bus_station']
print(bus_station.name.isna().sum(), bus_station.shape, bus_station.hex_10.nunique())
bus_station.head()

355 (1982, 369) 170


,element_type,osmid,amenity,created_by,name,geometry,addr:postcode,addr:street,country,name:en,name:hi,phone,website,barrier,ref,addr:housenumber,source,name:etymology:wikidata,wikidata,wikipedia,addr:housename,alt_name,addr:block,addr:place,addr:city,operator,building,addr:district,email,grades,religion,start_date,wikimedia_commons,office,opening_hours,description,shop,fax,phone_1,branch,wheelchair,isced:level,operator:type,addr:subdistrict,contact:phone,name:ru,contact:email,female,contact:fax,contact:website,short_name,brand,brand:wikidata,school:gender,survey:date,addr:suburb,note,addr:hamlet,contact:mobile,addr:full,check_date,nodes,name:ar,name:ml,name:zh,addr:country,landuse,postal_code,old_name,name:ur,male,internet_access,internet_access:fee,internet_access:ssid,name:ta,government,designation,is_in,boundary,addr:county,alt_name_1,int_name,tourism,not:brand:wikidata,name:fr,name:mr,addr:city:ur,leisure,addr:locality,building:levels,addr:unit,addr:neighbourhood,fixme,wall,level,building:colour,roof:shape,ways,admin_level,type,addr:sector,contact:facebook,dogs,internet_access:operator,not:name,surveillance,surveillance:type,usecase,denomination,name:de,name:es,name:he,name:it,name:pt,name:uk,building:material,height,is_in:city,is_in:country,is_in:country_code,is_in:iso_3166_2,length,name:bn,name:gu,name:ja,name:pa,name:te,width,name:fa,layer,access,fee,natural,sport,highway,bicycle,foot,facebook,key,lit,location,entrance,historic,motor_vehicle,community_centre:for,horse,brand:wikipedia,fitness_station,dance:teaching,unisex,landmark,air_conditioning,material,opening_hours:covid19,climbing:boulder,indoor,dog,location:transition,addr:khasra,image,contact:instagram,mobile,blind,garden:type,protect_class,surface,alt_name:hi,place,construction,disused:landuse,disused:name,disused:operator,covered,hoops,building:part,disused:amenity,water,grass,grassland,emergency,playground,fountain,addr:city:en,name:alt,lanes,oneway,segregated,smoking,source:boundary,area,residential,amenity_1,operator:short,operator:wikidata,playground:theme,roof:colour,swimming_pool,addr:province,addr:state,community_centre,notes,alt_name:ur,name:pnb,name:ro,colour,network,public_transport,railway,station,subway,name:kn,name:ks,train,network:wikidata,payment:cards,payment:cash,payment:coins,payment:contactless,payment:electronic_purses,alt_name:ar,alt_name:ks,name:az-Arab,name:azb,name:el,name:gr,name:grc,name:hy,name:hyw,name:ug,name_alt,long_name,long_name:ur,bus,depth,public_transport:version,toilets:wheelchair,bench,shelter,tactile_paving,wikipedia:ur,departures_board,operator:ur,line,elevation,official_name,depot,nohousenumber,popular name,payment:american_express,payment:amex,payment:credit_cards,payment:debit_cards,payment:diners_club,payment:discover_card,payment:maestro,payment:mastercard,payment:visa,payment:visa_debit,payment:visa_electron,diplomatic,embassy,target,roof:levels,building:levels:underground,cuisine,man_made,diet:lacto_vegetarian,diet:vegan,toilets,toilets:access,addr:street:ur,healthcare,healthcare:speciality,addr:floor,addr:place_1,phone_2,phone_3,health_facility:type,contact:nodal_officer_email,contact:nodal_officer_mobile,opening_hours:opd,emergency:phone,contact:youtube,addr:parentstreet,url,beds,loc_name,brand:pa,brand:ur,brand:wikipedia:pa,brand:wikipedia:ur,delivery,takeaway,brand:ks,indoor_seating,outdoor_seating,diet:vegetarian,wifi,alt_name:en,alt_name:pa,alt_name:pnb,brand:en,brand:hi,brand:pnb,official_name:pa,official_name:ur,drive_through,source:height,drink:coffee,drink:tea,seats,tables,payment:paytm,self_service,contact:snapchat,contact:twitter,happy_hours,alcohol,drink:hot_chocolate,organic,craft,cafe,fast_food,street_vendor,addr:place:ur,occupied,addr:plotnumber,informal,is_in:locality,atm,fuel:cng,payment:bhim,payment:mobikwik,payment:phonepe,payment:upi,compressed_air,fuel:diesel,fuel:octane_98,brand:website,source:amenity,fuel:petrol,fuel:octane_95,cash_withdrawal,brand:short,brand:kn,phone_4,branch:hi,branch:ty

### Railway Station

In [49]:
city_usecase1['network'].unique()

array([nan, 'Delhi Metro', 'IR', 'Delhi Public Transport', 'DTC',
       'Pink Line', 'DMRC'], dtype=object)

In [50]:
railway_station = city_usecase1[city_usecase1['network'] == 'IR']
print(railway_station.name.isna().sum(), railway_station.shape, railway_station.hex_10.nunique())
railway_station.head()

0 (56, 369) 39


,element_type,osmid,amenity,created_by,name,geometry,addr:postcode,addr:street,country,name:en,name:hi,phone,website,barrier,ref,addr:housenumber,source,name:etymology:wikidata,wikidata,wikipedia,addr:housename,alt_name,addr:block,addr:place,addr:city,operator,building,addr:district,email,grades,religion,start_date,wikimedia_commons,office,opening_hours,description,shop,fax,phone_1,branch,wheelchair,isced:level,operator:type,addr:subdistrict,contact:phone,name:ru,contact:email,female,contact:fax,contact:website,short_name,brand,brand:wikidata,school:gender,survey:date,addr:suburb,note,addr:hamlet,contact:mobile,addr:full,check_date,nodes,name:ar,name:ml,name:zh,addr:country,landuse,postal_code,old_name,name:ur,male,internet_access,internet_access:fee,internet_access:ssid,name:ta,government,designation,is_in,boundary,addr:county,alt_name_1,int_name,tourism,not:brand:wikidata,name:fr,name:mr,addr:city:ur,leisure,addr:locality,building:levels,addr:unit,addr:neighbourhood,fixme,wall,level,building:colour,roof:shape,ways,admin_level,type,addr:sector,contact:facebook,dogs,internet_access:operator,not:name,surveillance,surveillance:type,usecase,denomination,name:de,name:es,name:he,name:it,name:pt,name:uk,building:material,height,is_in:city,is_in:country,is_in:country_code,is_in:iso_3166_2,length,name:bn,name:gu,name:ja,name:pa,name:te,width,name:fa,layer,access,fee,natural,sport,highway,bicycle,foot,facebook,key,lit,location,entrance,historic,motor_vehicle,community_centre:for,horse,brand:wikipedia,fitness_station,dance:teaching,unisex,landmark,air_conditioning,material,opening_hours:covid19,climbing:boulder,indoor,dog,location:transition,addr:khasra,image,contact:instagram,mobile,blind,garden:type,protect_class,surface,alt_name:hi,place,construction,disused:landuse,disused:name,disused:operator,covered,hoops,building:part,disused:amenity,water,grass,grassland,emergency,playground,fountain,addr:city:en,name:alt,lanes,oneway,segregated,smoking,source:boundary,area,residential,amenity_1,operator:short,operator:wikidata,playground:theme,roof:colour,swimming_pool,addr:province,addr:state,community_centre,notes,alt_name:ur,name:pnb,name:ro,colour,network,public_transport,railway,station,subway,name:kn,name:ks,train,network:wikidata,payment:cards,payment:cash,payment:coins,payment:contactless,payment:electronic_purses,alt_name:ar,alt_name:ks,name:az-Arab,name:azb,name:el,name:gr,name:grc,name:hy,name:hyw,name:ug,name_alt,long_name,long_name:ur,bus,depth,public_transport:version,toilets:wheelchair,bench,shelter,tactile_paving,wikipedia:ur,departures_board,operator:ur,line,elevation,official_name,depot,nohousenumber,popular name,payment:american_express,payment:amex,payment:credit_cards,payment:debit_cards,payment:diners_club,payment:discover_card,payment:maestro,payment:mastercard,payment:visa,payment:visa_debit,payment:visa_electron,diplomatic,embassy,target,roof:levels,building:levels:underground,cuisine,man_made,diet:lacto_vegetarian,diet:vegan,toilets,toilets:access,addr:street:ur,healthcare,healthcare:speciality,addr:floor,addr:place_1,phone_2,phone_3,health_facility:type,contact:nodal_officer_email,contact:nodal_officer_mobile,opening_hours:opd,emergency:phone,contact:youtube,addr:parentstreet,url,beds,loc_name,brand:pa,brand:ur,brand:wikipedia:pa,brand:wikipedia:ur,delivery,takeaway,brand:ks,indoor_seating,outdoor_seating,diet:vegetarian,wifi,alt_name:en,alt_name:pa,alt_name:pnb,brand:en,brand:hi,brand:pnb,official_name:pa,official_name:ur,drive_through,source:height,drink:coffee,drink:tea,seats,tables,payment:paytm,self_service,contact:snapchat,contact:twitter,happy_hours,alcohol,drink:hot_chocolate,organic,craft,cafe,fast_food,street_vendor,addr:place:ur,occupied,addr:plotnumber,informal,is_in:locality,atm,fuel:cng,payment:bhim,payment:mobikwik,payment:phonepe,payment:upi,compressed_air,fuel:diesel,fuel:octane_98,brand:website,source:amenity,fuel:petrol,fuel:octane_95,cash_withdrawal,brand:short,brand:kn,phone_4,branch:hi,branch:ty

### Fare Estimates

In [51]:
cwd = os.getcwd()
cwd

'/Users/rapido/analytics/customer/SEO'

In [701]:
local_datasets = '/Users/rapido/local-datasets/customer/seo/raw/'

In [56]:
city='Delhi'
start='20231106'
end='20231110'

fe_rr_query=f"""
    with base as (
        select 
            customer_id, taxi_lifetime_last_ride_city, taxi_lifetime_stage, 
            taxi_retention_segments, taxi_gc_tag,
            taxi_time_of_day_preference, taxi_day_of_week_preference
        from 
            datasets.iallocator_customer_segments
        where 
            run_date = date_format(date_parse('{end}','%Y%m%d') + interval '1' day, '%Y-%m-%d')
            and taxi_lifetime_last_ride_city = '{city}'
            -- and taxi_lifetime_last_ride_city in ('Delhi','New Delhi','Noida','Gurugram')
            -- and taxi_lifetime_stage='COMMITTED'
    ),

    base_fe_address_id as (
        select 
            *
        from
        (
            select 
                eventprops_pickup_place_id as pickup_place_id, 
                eventprops_drop_place_id as drop_place_id, 
                pickup_address as pickup_address, 
                drop_address as drop_address,
                identity as customer_id,
                quarter_hour, yyyymmdd, sub_total, eta,
                fare_estimate_id as fe_estimate_id, 
                current_city as city, city as city2,
                service_name as service, 
                ct_session_id as eventprops_ctsessionid,
                pickup_latitude as pickup_lat,
                pickup_longitude as pickup_lng,
                drop_location_latitude as drop_lat,
                drop_location_longitude as drop_lng,
                pickup_location_hex_12 as pickup_hex_12,
                drop_location_hex_12 as drop_hex_12,
                drop_location_hex_8_cluster as drop_cluster,
                pickup_location_hex_8_cluster as pickup_cluster,
                row_number() over (partition by cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone order by epoch desc) as fe_updated_seq,
                cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone AS unique_id
            from 
                canonical.clevertap_customer_fare_estimate
            where 
                yyyymmdd >= '{start}'
                and yyyymmdd <= '{end}'
                and current_city = '{city}'
                -- and current_city in ('Delhi','Gurugram','New Delhi','Noida')
                and service_name in ('Auto','Link','Bike','CabEconomy')
        )
        where fe_updated_seq=1
    ),

    base_rr_address_id as (
        select 
            *
        from
        (
            select 
                pickup_address eventprops_pickaddress, drop_address as eventprops_dropaddress, 
                user_id as customer_id, yyyymmdd, fare_estimate_id as rr_estimate_id, 
                service_name  as eventprops_servicename, 
                ct_session_id as eventprops_ctsessionid,
                pickup_location_latitude as eventprops_picklat, pickup_location_longitude as eventprops_picklng,
                row_number() over (partition by cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone order by epoch desc) as rr_updated_seq,
                cast(cast(ct_session_id as decimal) as varchar) || ' - ' || phone AS unique_id
            from 
                canonical.clevertap_customer_request_rapido
            where 
                yyyymmdd >= '{start}'
                and yyyymmdd <= '{end}'
                and current_city = '{city}'
                and service_name in ('Auto','Link','Bike','CabEconomy')
        )
        where rr_updated_seq=1
    ),
    
    final as (
    select 
        fe.*, rr.rr_estimate_id, 
        rr.eventprops_servicename as rr_service, rr.unique_id as rrs
    from
        base_fe_address_id fe
    left join
        base_rr_address_id rr
        on fe.unique_id = rr.unique_id
    )

    select
        a.*,
        b.*
    from 
        final a 
    left join 
        base b 
        on a.customer_id = b.customer_id
"""

usecase_fe_rr_cust_new =pd.read_sql(fe_rr_query, connection)
usecase_fe_rr_cust_new.to_csv(local_datasets + 'usecase_fe_rr_cust_{}.csv'.format(city), index=False)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_1309/1116151249.py:103: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  usecase_fe_rr_cust_new =pd.read_sql(fe_rr_query, connection)


DatabaseError: Execution failed on sql: 
    with base as (
        select 
            customer_id, taxi_lifetime_last_ride_city, taxi_lifetime_stage, 
            taxi_retention_segments, taxi_gc_tag,
            taxi_time_of_day_preference, taxi_day_of_week_preference
        from 
            datasets.iallocator_customer_segments
        where 
            run_date = date_format(date_parse('20231110','%Y%m%d') + interval '1' day, '%Y-%m-%d')
            and taxi_lifetime_last_ride_city = 'Delhi'
            -- and taxi_lifetime_last_ride_city in ('Delhi','New Delhi','Noida','Gurugram')
            -- and taxi_lifetime_stage='COMMITTED'
    ),

    base_fe_address_id as (
        select 
            *
        from
        (
            select 
                eventprops_pickup_place_id as pickup_place_id, 
                eventprops_drop_place_id as drop_place_id, 
                pickup_address as pickup_address, 
                drop_address as drop_address,
                identi

In [703]:
usecase_fe_rr_cust_new = pd.read_csv(local_datasets + 'usecase_fe_rr_cust_{}.csv'.format(city))

In [704]:
usecase_fe_rr_cust_new.head()

,pickup_place_id,drop_place_id,pickup_address,drop_address,customer_id,quarter_hour,yyyymmdd,sub_total,eta,fe_estimate_id,city,city2,service,eventprops_ctsessionid,pickup_lat,pickup_lng,drop_lat,drop_lng,pickup_hex_12,drop_hex_12,drop_cluster,pickup_cluster,fe_updated_seq,unique_id,rr_estimate_id,rr_service,rrs,customer_id.1,taxi_lifetime_last_ride_city,taxi_lifetime_stage,taxi_retention_segments,taxi_gc_tag,taxi_time_of_day_preference,taxi_day_of_week_preference
0,NaN,ChIJ93ZVg9y3bTkRTHmNXY79ESg,"103 104, Purusharth Nagar C-2, Mithila vihar, ...","Jagatpura railway station platform No 2, Railw...",64f9d7e50fc9ea3f551efdcb,1745,20231109,26.0,0.0,654cd069bbd2b41fe80e2c87,Jaipur,Jaipur,Link,1.699533e+09,26.830630,75.833540,26.836992,75.831830,8c3da20b18e27ff,8c3da20a25939ff,Jhalana Park_JAI,Siddharth Nagar_JAI,1,1699532798 - 918058814929,654cd069bbd2b41fe80e2c87,Auto,1699532798 - 918058814929,64f9d7e50fc9ea3f551efdcb,Jaipur,HOOK,GOLD,UNKNOWN,EVENING_PEAK,UNKNOWN
1,ChIJTdYovAy2bTkRhsAcvCUAjjI,ChIJRyDM9AS1bTkRNLH-yaVg6Vw,"4/390, Vidyut Abhiyanta Colony, Sector 4, Malv...","Gandhi Nagar Railway Station Jaipur, Jawahar N...",64745fead8722e70c405283f,1200,20231108,48.0,2.0,654b2e45523d3e2754da3ab7,Jaipur,Jaipur,Link,1.699421e+09,26.850245,75.813790,26.872274,75.799160,8c3da20a64a5bff,8c3da219902bdff,Gopalpura_JAI,Satkar_Malviya_Nagar_JAI,1,1699420712 - 918529839136,654b244a130a3a01eb9c74a5,Auto,1699420712 - 918529839136,64745fead8722e70c405283f,Jaipur,HANDHOLDING,HH,UNKNOWN,MORNING_PEAK,WEEKDAY
2,NaN,NaN,"Entry and Exit Gate , Poornima Engineering col...","Plot No.-70, UG-5,Braj Bhawan Opposite ICICI B...",62305f98e33a230036a148b9,1115,20231109,352.0,1.0,654c74aef080094cf88c3fd1,Jaipur,Jaipur,Auto,1.699509e+09,26.768576,75.850660,26.943590,75.748795,8c3da20900a41ff,8c3da21ae6e6dff,Jhotwara_JAI,Sitapura_JAI,1,1699509407 - 919358590309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ChIJIydFzMS1bTkRAIqxi6BaHIg,ChIJGXfwUfizbTkRIcHXzAUkPMA,"Jaipur Hospital, Mahaveer Nagar, Tonk Road, ne...","Jaipur International Airport (JAI), Airport Ro...",6549f7b51e0096f10567f917,1145,20231108,68.0,2.0,654b277c860e1f7d70433865,Jaipur,Jaipur,Link,1.699424e+09,26.857447,75.794975,26.828945,75.805620,8c3da219b84a7ff,8c3da20b42ec5ff,Airport_JAI,Malviya_Nagar_JAI 2,1,1699423873 - 919040606436,654b277c860e1f7d70433865,Auto,1699423873 - 919040606436,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,ChIJe0jKQUOzbTkRL6h22X4aS4A,"NH11C, Sodala, Jaipur, Rajasthan 302011, India","Manappuram Finance Limited, Subash Nagar, Shas...","5d6b84efd0286d106d710b53,62815333dc46f32934204d52",845,20231109,63.0,1.0,654c4fd7fd98c85c9aac0267,Jaipur,Jaipur,Link,1.699500e+09,26.898710,75.766290,26.936623,75.795570,8c3da21833563ff,8c3da218dd095ff,Subhash Nagar,Nirman_Nagar_JAI,1,1699499938 - 918109385062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [705]:
usecase_fe_rr_cust_new['drop_hex_10'] = usecase_fe_rr_cust_new[['drop_lat', 'drop_lng']].apply(lambda x: h3.geo_to_h3(x[0], x[1], resolution=10), axis=1)
usecase_fe_rr_cust_new['pickup_hex_10'] = usecase_fe_rr_cust_new[['pickup_lat', 'pickup_lng']].apply(lambda x: h3.geo_to_h3(x[0], x[1], resolution=10), axis=1)

In [706]:
fes = usecase_fe_rr_cust_new.groupby(['drop_hex_10']).agg(
tot_fe = ('fe_estimate_id', 'nunique'),
lat = ('drop_lat', 'mean'),
lng = ('drop_lng', 'mean'),

).reset_index()
fes.head()

,drop_hex_10,tot_fe,lat,lng
0,0,14961,NaN,NaN
1,8a3ce15602cffff,1,26.189508,91.772370
2,8a3cf2d59a07fff,0,22.583000,88.337290
3,8a3d10d2351ffff,0,30.708206,76.847570
4,8a3d14693147fff,0,30.707682,76.712074


## Data

### Metro Data

In [707]:
metro.head()
metro1 = metro[['osmid', 'name', 'hex_12_list', 'hex_10', 'hex_8']]

In [708]:
metro2 = metro1.merge(fes, left_on='hex_10', right_on='drop_hex_10', how='left')
metro2.head(2)

,osmid,name,hex_12_list,hex_10,hex_8,drop_hex_10,tot_fe,lat,lng
0,1064270892,Durgapura Metro (u/c),8c3da20b4ba33ff,8a3da20b4ba7fff,883da20b4bfffff,8a3da20b4ba7fff,13,26.835791,75.793221
1,1064270917,Ambabadi Metro (u/c),8c3da21ab5a35ff,8a3da21ab5a7fff,883da21ab5fffff,8a3da21ab5a7fff,11,26.940382,75.777592


In [709]:
final_data_metro = metro2\
                    .groupby(['name'])\
                    .agg(
                        hex_10_list = ('hex_10', 'unique'),
                        tot_fe = ('tot_fe', 'sum'),
                        lat = ('lat', 'mean'),
                        lng = ('lng', 'mean')
                        )\
                    .reset_index()
final_data_metro['transport_type'] = 'Metro'
final_data_metro.head(2)

,name,hex_10_list,tot_fe,lat,lng,transport_type
0,Ambabadi Metro (u/c),[8a3da21ab5a7fff],11,26.940382,75.777592,Metro
1,Badi Chaupar,[8a3da2189007fff],537,26.922958,75.827078,Metro


### Bus Station Data

In [710]:
bus_station1 = bus_station[['osmid', 'name', 'hex_12_list', 'hex_10', 'hex_8']]

In [711]:
bus_station2 = bus_station1.merge(fes, left_on='hex_10', right_on='drop_hex_10', how='left')
bus_station2.head(2)

,osmid,name,hex_12_list,hex_10,hex_8,drop_hex_10,tot_fe,lat,lng
0,862954948,Sanganer Bus Depot,8c3da20b205d9ff,8a3da20b205ffff,883da20b21fffff,8a3da20b205ffff,12.0,26.795606,75.815190
1,862954948,Sanganer Bus Depot,8c3da20b2048bff,8a3da20b204ffff,883da20b21fffff,8a3da20b204ffff,35.0,26.796378,75.815394


In [712]:
final_data_bus_station = bus_station2\
                            .groupby(['name'])\
                            .agg(
                                hex_10_list = ('hex_10', 'unique'),
                                tot_fe = ('tot_fe', 'sum'),
                                lat = ('lat', 'mean'),
                                lng = ('lng', 'mean')
                                )\
                            .reset_index()

final_data_bus_station['transport_type'] = 'Bus station'
final_data_bus_station.head(2)

,name,hex_10_list,tot_fe,lat,lng,transport_type
0,22 Godam Public Bus Stop,[8a3da218a92ffff],278.0,26.902363,75.793459,Bus station
1,Chomu Pulia,[8a3da21aa35ffff],1292.0,26.941526,75.773461,Bus station


### Railway Station Data

In [713]:
railway_station1 = railway_station[['osmid', 'name', 'hex_12_list', 'hex_10', 'hex_8']]

In [714]:
railway_station2 = railway_station1.merge(fes, left_on='hex_10', right_on='drop_hex_10', how='left')
railway_station2.head(2)

,osmid,name,hex_12_list,hex_10,hex_8,drop_hex_10,tot_fe,lat,lng
0,316681502,Dahar-Ka-Balaji,8c3da21aaaacdff,8a3da21aaaaffff,883da21aabfffff,8a3da21aaaaffff,291.0,26.944319,75.766085
1,1753726590,Gandhinagar Jaipur,8c3da219915ebff,8a3da219915ffff,883da21991fffff,8a3da219915ffff,690.0,26.873619,75.799237


In [715]:
final_data_railway_station = railway_station2\
                            .groupby(['name'])\
                            .agg(
                                hex_10_list = ('hex_10', 'unique'),
                                tot_fe = ('tot_fe', 'sum'),
                                lat = ('lat', 'mean'),
                                lng = ('lng', 'mean')
                                )\
                            .reset_index()

final_data_railway_station['transport_type'] = 'Railway station'
final_data_railway_station.head(2)

,name,hex_10_list,tot_fe,lat,lng,transport_type
0,AsaIpur Jobner,[8a3da2c024e7fff],0.0,NaN,NaN,Railway station
1,Badhal,[8a3da2984a47fff],0.0,NaN,NaN,Railway station


### Output 

In [716]:
local_datasets

'/Users/rapido/local-datasets/customer/seo/raw/'

In [717]:
df_final_data = [final_data_metro, final_data_bus_station, final_data_railway_station]
final_data = pd.concat(df_final_data)
final_data['city'] = city
final_data

,name,hex_10_list,tot_fe,lat,lng,transport_type,city
0,Ambabadi Metro (u/c),[8a3da21ab5a7fff],11.0,26.940382,75.777592,Metro,Jaipur
1,Badi Chaupar,[8a3da2189007fff],537.0,26.922958,75.827078,Metro,Jaipur
2,Chandpole,[8a3da218d797fff],195.0,26.926381,75.807513,Metro,Jaipur
3,Choti Chaupar,[8a3da2189d5ffff],651.0,26.924678,75.818514,Metro,Jaipur
4,Civil Lines,[8a3da218e16ffff],135.0,26.909447,75.781071,Metro,Jaipur
5,Durgapura Metro (u/c),[8a3da20b4ba7fff],13.0,26.835791,75.793221,Metro,Jaipur
6,Manasarovar,[8a3da219c8f7fff],951.0,26.879435,75.750962,Metro,Jaipur
7,Mansarovar Metro Station,"[8a3da219c8f7fff, 8a3da219c8affff]",5724.0,26.879402,75.750775,Metro,Jaipur
8,New Aatish Market,[8a3da219dd97fff],201.0,26.880269,75.764521,Metro,Jaipur
9,Railway Station,[8a3da218c6f7fff],764.0,26.918015,75.790034,Metro,Jaipur


In [718]:
final_data.to_csv('/Users/rapido/local-datasets/customer/seo/processed/' 
                  + 
                  'data_metro_railway_bus_station_{}.csv'.format(city), index=False)

In [719]:
final_data.to_clipboard(index=False)